# Exercise: Controlling Output Shape with Stop Sequences and Prefill

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and rewrote cloud-specific prompts to **Azure**.

## The exercise

Two control surfaces combine here to force Claude's output into an exact shape:

- **`stop_sequences`** halts generation the moment Claude emits a matching string. Here the trigger is the closing triple-backtick, so Claude stops right after the fenced block.
- **Assistant prefill** seeds the start of Claude's turn. By ending the prefill with an opening ` ```bash ` fence, Claude is committed to continuing inside that block, so it emits **only the three Azure CLI commands** with no preamble, no numbering, no trailing prose.

The goal: produce three short **Azure CLI** commands as raw shell text, then `.strip()` the whitespace. The steps below follow the same preamble as every notebook in this folder (**install**, **load `.env`**, **create client**), then define helpers and run the forced-shape request.

> The prefilled fence never comes back in the response - the API returns only what Claude generates AFTER the prefill. Reattach the opening fence yourself if you want a complete code block for display.

### 1. Install dependencies

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

`add_user_message` and `add_assistant_message` append role-tagged turns to a `messages` list. The **assistant prefill** is just an assistant turn added BEFORE the request, so its text becomes the opening of Claude's reply. `chat` forwards optional `system`, `temperature`, and **`stop_sequences`** into `client.messages.create`.

In [4]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

### 5. Force three Azure CLI commands with prefill plus stop sequence

The **user prompt** requests three short **Azure CLI** commands. The **assistant prefill** ends with an opening ` ```bash ` fence, so Claude must continue inside a shell block. The **`stop_sequences=["```"]`** argument halts generation at the closing fence, so nothing trails after the third command. `.strip()` trims the leading and trailing whitespace from the returned raw text.

In [5]:
messages = []

prompt = """
Generate three different sample Azure CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments: ```bash")

text = chat(messages, stop_sequences=["```"])

text.strip()



'az group create --name myResourceGroup --location eastus\naz storage account create --name mystorageacct --resource-group myResourceGroup --location eastus --sku Standard_LRS\naz vm create --resource-group myResourceGroup --name myVM --image UbuntuLTS --admin-username azureuser'